In [11]:
import mdtraj as md

In [12]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
fixer = PDBFixer(filename='ACTB_cryo.pdb')
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(False)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
PDBFile.writeFile(fixer.topology, fixer.positions, open('output.pdb', 'w'))

In [13]:
traj = md.load('output.pdb')

In [14]:
traj = traj.atom_slice(traj.topology.select('element != H'))
traj

<mdtraj.Trajectory with 1 frames, 2927 atoms, 375 residues, without unitcells at 0x152056f4f100>

In [15]:
old_top = traj.topology
new_top = md.Topology()

# Create new topology with renumbered atoms/residues
chain_map = {}
residue_map = {}

for chain in old_top.chains:
    new_chain = new_top.add_chain()
    chain_map[chain] = new_chain

    for res in chain.residues:
        new_res = new_top.add_residue(res.name, new_chain)
        residue_map[res] = new_res

        for atom in res.atoms:
            new_top.add_atom(atom.name, atom.element, new_res)

# Create new trajectory with same coordinates
new_traj = md.Trajectory(xyz=traj.xyz, topology=new_top)

In [16]:
new_traj.save('actb_noH.pdb', force_overwrite=True)